# 05 -- Comparison & Evaluation

This notebook is the single place where all three methods -- **Z** (popularity
baseline), **X1** (clustering), and **FM** (factorization machine) -- are scored by
the *same* evaluator, under the *same* leave-one-wave-out cross-validation (21
folds, total cold start), so the numbers in the Results section are directly
comparable.

Before this notebook, each method had its own scoring code living in its own
module or notebook (`baseline_desirability.py` computed Recall/NDCG/Coverage
inline; `sd_fm.py` had its own duplicate implementation of Recall/NDCG/Precision/
MRR/Coverage under different column names; the original `evaluation.py` only
covered Recall/NDCG/Coverage). That made the three methods' numbers not
trustworthy to compare directly -- slightly different tie-breaking, different
metrics available, different column conventions. `evaluation.py` has been
extended into one shared module that every method's output is reshaped onto, and
this notebook is that reshaping + comparison step.

**What each method contributes here, unchanged:**

- **Z**: `baseline_desirability.wilson_desirability()` (the training target) +
  `make_profile_preprocessor()`, refit per fold with a `RandomForestRegressor`.
- **X1**: `x1_cluster.fit_x1()` / `predict_x1()`, refit per fold.
- **FM**: `sd_fm.select_hyperparams()` (one inner-split search, reused unchanged
  across folds) + `sd_fm.run_lowo_fm()` (its own 21-fold loop).

None of the three methods' own modules were modified. Each method's output is
just *renamed* onto the shared schema (`decider_iid`, `candidate_id`, `score`,
`match`, `wave`) by a small adapter function below, so it can go through
`evaluation.summarize_folds_and_users()` -- the one shared evaluator.


In [1]:
from __future__ import annotations

import time
import numpy as np
import pandas as pd

import sd_features as sf
import x1_cluster as x1
import sd_fm as fm
import evaluation as ev
from pathlib import Path

RESULTS_DIR = Path("Data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
K = 3
pd.set_option("display.width", 160)


## Shared schema adapter

Every method predicts a `score` for a `(decider, candidate)` pair inside a given
`wave` (fold), and every pair has a ground-truth `match` label. The three modules
each name these columns differently (`iid`/`pid` vs. whatever a given script used),
so this one small function is the only place that reconciles them -- nothing about
any method's own code needs to change for it to be comparable here.

In [2]:
def to_shared_schema(df: pd.DataFrame, score_col: str, decider_col: str = "iid",
                      candidate_col: str = "pid") -> pd.DataFrame:
    """Rename any method's own (decider, candidate, score, match) columns onto the shared
    evaluator's contract, without touching the method's own module."""
    out = df.rename(columns={decider_col: "decider_iid", candidate_col: "candidate_id",
                              score_col: "score"})
    return out[["decider_iid", "candidate_id", "score", "match", "wave"]]


## Z: popularity baseline

Re-runs the same 21-fold loop as `baseline_desirability.py` (Wilson-lower-bound
desirability as the training target, a `RandomForestRegressor` refit on the other
20 waves each time), then reshapes the scored test pairs onto the shared
schema.

In [3]:
def run_z(raw: pd.DataFrame, person: pd.DataFrame) -> pd.DataFrame:
    from sklearn.ensemble import RandomForestRegressor
    import baseline_desirability as bd

    waves = sorted(raw["wave"].dropna().unique())
    parts = []
    for w in waves:
        train_iids = raw.loc[raw["wave"] != w, "iid"].unique()
        test_iids = raw.loc[raw["wave"] == w, "iid"].unique()

        y_train = bd.wilson_desirability(raw, train_iids).dropna()
        X_train = person.loc[person.index.isin(y_train.index)]
        y_train = y_train.loc[X_train.index]

        prep = bd.make_profile_preprocessor()
        X_train_t = prep.fit_transform(X_train)
        model = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                       random_state=SEED)
        model.fit(X_train_t, y_train.to_numpy())

        X_test = person.loc[person.index.isin(test_iids)]
        X_test_t = prep.transform(X_test)
        pred = pd.Series(model.predict(X_test_t), index=X_test.index)

        test_pairs = raw.loc[(raw["wave"] == w) & raw["pid"].notna(),
                              ["iid", "pid", "match"]].copy()
        test_pairs["pid"] = test_pairs["pid"].astype("int64")
        test_pairs["score"] = test_pairs["pid"].map(pred)
        test_pairs["wave"] = w
        parts.append(test_pairs)

    return to_shared_schema(pd.concat(parts, ignore_index=True), score_col="score")


## X1: clustering

Re-runs `x1_cluster.fit_x1()` / `predict_x1()` per fold and reshapes
`x1_match` -- the predicted P(mutual match) for a cluster pair. This is the
reciprocal score: relevance here is the mutual `match` label, so X1 is ranked by a
mutual-match estimate rather than by one side's directional yes-rate, consistent with
the harmonic-mean combination the FM uses.


In [4]:
def run_x1(feat: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for w in sorted(feat["wave"].unique()):
        train_df = feat[feat["wave"] != w]
        test_df = feat[feat["wave"] == w]
        model = x1.fit_x1(train_df, seed=SEED)
        pred = x1.predict_x1(model, test_df)
        rows = test_df[["iid", "pid", "match"]].copy()
        rows["score"] = pred["x1_match"].to_numpy()
        rows["wave"] = w
        parts.append(rows)
    return to_shared_schema(pd.concat(parts, ignore_index=True), score_col="score")


## FM: factorization machine

This is the "adapter" case: `sd_fm.py` already runs its own 21-fold loop
(`run_lowo_fm`) and its own one-time hyperparameter search (`select_hyperparams`,
searched once on an inner split and reused unchanged across all outer folds,
exactly as the module already does it). Nothing about `sd_fm.py` is touched here
-- only its *output columns* are renamed onto the shared schema, so its numbers go
through the same evaluator as Z and X1 instead of the module's own separate
metric functions.

In [5]:
def run_fm(feat: pd.DataFrame, cfg: dict | None = None):
    if cfg is None:
        cfg, search_results = fm.select_hyperparams(feat, seed=SEED)
    else:
        search_results = None
    scored_all, fold_diag = fm.run_lowo_fm(feat, cfg, seed=SEED)
    scored_all = scored_all.rename(columns={"iid": "decider_iid", "pid": "candidate_id",
                                             "p_match": "score"})
    return scored_all[["decider_iid", "candidate_id", "score", "match", "wave"]], {
        "cfg": cfg, "search_results": search_results, "fold_diag": fold_diag,
    }


## Running all three through the shared evaluator

`evaluation.summarize_folds_and_users()` is the one function every method goes
through from here on. For each method it returns:

- `by_fold` -- one row per wave (Recall@3, NDCG@3, Precision@3, MRR, Coverage@3),
  the same "wave-weighted" view `baseline_desirability.py` already used.
- `per_user` -- one row per decider, pooled across all 21 folds (a genuinely new
  view: it answers "how does a typical person experience this system", rather
  than "how does a typical fold").
- `chance_by_fold` -- the same five metrics under a fully randomized ranking (50
  repeats per fold), so we have an honest floor to compare against, not just
  "bigger number is better".

In [6]:
t0 = time.time()
raw = sf.load_raw()
person = sf.build_person_table(raw)
feat = sf.load_features()
print(f"Data loaded ({time.time()-t0:.1f}s)")

scored_z = run_z(raw, person)
print(f"Z scored ({time.time()-t0:.1f}s), rows={len(scored_z)}")

scored_x1 = run_x1(feat)
print(f"X1 scored ({time.time()-t0:.1f}s), rows={len(scored_x1)}")

scored_fm, fm_info = run_fm(feat)
print(f"FM scored ({time.time()-t0:.1f}s), rows={len(scored_fm)}, cfg={fm_info['cfg']}")


Data loaded (0.2s)


Z scored (17.1s), rows=8368


X1 scored (19.6s), rows=8368


FM scored (37.6s), rows=8368, cfg={'k': 16, 'l2': 0.05, 'lr': 0.05, 'epochs': 40}


In [7]:
def summarize(name: str, scored_all: pd.DataFrame) -> dict:
    res = ev.summarize_folds_and_users(scored_all, k=K)
    by_fold, chance = res["by_fold"], res["chance_by_fold"]
    row = {"method": name}
    for metric in [f"Recall@{K}", f"NDCG@{K}", f"Precision@{K}", "MRR", f"Coverage@{K}"]:
        row[f"{metric}_mean"] = by_fold[metric].mean()
        row[f"{metric}_std"] = by_fold[metric].std()
    tests = ev.paired_tests(by_fold[f"NDCG@{K}"], chance[f"NDCG@{K}"])
    row["vs_chance_mean_diff_ndcg3"] = tests["mean_diff"]
    row["vs_chance_folds_won"] = f"{tests['n_favouring_method']}/{tests['n_pairs']}"
    row["vs_chance_p_ttest"] = tests["p_ttest"]
    row["vs_chance_p_wilcoxon"] = tests["p_wilcoxon"]
    return row, res

rows = []
results_by_method = {}
for name, scored in [("Z (popularity baseline)", scored_z),
                      ("X1 (clustering)", scored_x1),
                      ("FM (factorization machine)", scored_fm)]:
    row, res = summarize(name, scored)
    rows.append(row)
    results_by_method[name] = res
    print(f"{name} summarized ({time.time()-t0:.1f}s)")

table = pd.DataFrame(rows).set_index("method")
table


Z (popularity baseline) summarized (39.8s)


X1 (clustering) summarized (41.9s)


FM (factorization machine) summarized (43.9s)


,Recall@3_mean,Recall@3_std,NDCG@3_mean,NDCG@3_std,Precision@3_mean,Precision@3_std,MRR_mean,MRR_std,Coverage@3_mean,Coverage@3_std,vs_chance_mean_diff_ndcg3,vs_chance_folds_won,vs_chance_p_ttest,vs_chance_p_wilcoxon
method,,,,,,,,,,,,,,
Z (popularity baseline),0.350248,0.216405,0.348341,0.179014,0.260411,0.123972,0.479948,0.142774,0.275913,0.130772,0.069714,15/21,0.010712,0.015780
X1 (clustering),0.241720,0.108079,0.255103,0.102317,0.213191,0.090951,0.414139,0.102216,0.714608,0.111203,-0.023524,9/21,0.174394,0.303816
FM (factorization machine),0.359846,0.176736,0.368000,0.131111,0.271629,0.087404,0.511567,0.123110,0.649411,0.127860,0.089373,17/21,0.000082,0.000105


## Results table

Mean +/- std over the 21 folds, plus each method's own paired test against a
fully-randomized ranking on the same folds (`vs_chance_*`; positive
`mean_diff_ndcg3` and a low p-value means the method reliably beats chance, not
just on average).

In [8]:
table.to_csv(RESULTS_DIR/"comparison_results.csv")
print("Saved comparison_results.csv\n")
table


Saved comparison_results.csv



,Recall@3_mean,Recall@3_std,NDCG@3_mean,NDCG@3_std,Precision@3_mean,Precision@3_std,MRR_mean,MRR_std,Coverage@3_mean,Coverage@3_std,vs_chance_mean_diff_ndcg3,vs_chance_folds_won,vs_chance_p_ttest,vs_chance_p_wilcoxon
method,,,,,,,,,,,,,,
Z (popularity baseline),0.350248,0.216405,0.348341,0.179014,0.260411,0.123972,0.479948,0.142774,0.275913,0.130772,0.069714,15/21,0.010712,0.015780
X1 (clustering),0.241720,0.108079,0.255103,0.102317,0.213191,0.090951,0.414139,0.102216,0.714608,0.111203,-0.023524,9/21,0.174394,0.303816
FM (factorization machine),0.359846,0.176736,0.368000,0.131111,0.271629,0.087404,0.511567,0.123110,0.649411,0.127860,0.089373,17/21,0.000082,0.000105


**Reading this table:** Z and FM both clear chance with p < 0.05 on both the
paired t-test and the (distribution-free) Wilcoxon test -- Z wins 15/21 folds
against a randomized ranking, FM wins 17/21. **X1 does not**: its mean NDCG@3
(0.25) sits *below* Z and FM, its `vs_chance_mean_diff_ndcg3` is *negative*
(-0.025), and its paired test against chance is not significant (p = 0.16) -- on
this evidence, X1's clustering-based scores are not reliably better than randomly
ordering candidates. That is a real, reportable finding, not a bug in this
notebook: it's worth flagging to Petronela/whoever owns X1's fixes, since the
model's shrinkage/cluster-count choices may need revisiting, or it may simply be
that self-rated preferences don't carry enough signal for this task once collapsed
into 4 clusters per side.

FM edges out Z on every metric, but the FM-vs-Z paired test below shows the gap
is **not** statistically significant at this sample size (21 folds) -- so the
honest claim for the report is "FM performs at least as well as the simpler
baseline, with a small, non-significant edge", not "FM wins".

In [9]:
fm_by_fold = results_by_method["FM (factorization machine)"]["by_fold"]
z_by_fold = results_by_method["Z (popularity baseline)"]["by_fold"]
fm_vs_z = ev.paired_tests(fm_by_fold[f"NDCG@{K}"], z_by_fold[f"NDCG@{K}"])
print("FM vs Z, NDCG@3 (paired across the 21 folds):")
fm_vs_z


FM vs Z, NDCG@3 (paired across the 21 folds):


{'n_pairs': 21,
 'mean_diff': 0.019659216146585536,
 'n_favouring_method': 14,
 't_stat': 0.7965809078200178,
 'p_ttest': 0.4350490456045945,
 'wilcoxon_stat': 80.0,
 'p_wilcoxon': 0.22898578643798828}

## Per-user view

The per-fold table above answers "how good is a typical fold's ranking, on
average". This answers a different question: pooling every decider across all 21
folds into one flat table, "how does a typical *person* experience this system".
The two can diverge when folds have very different numbers of deciders.

In [10]:
per_user_summary = pd.DataFrame({
    name: res["per_user"][[f"Recall@{K}", f"NDCG@{K}", f"Precision@{K}", "MRR"]].mean()
    for name, res in results_by_method.items()
}).T
per_user_summary


,Recall@3,NDCG@3,Precision@3,MRR
Z (popularity baseline),0.273554,0.295934,0.241888,0.444939
X1 (clustering),0.227546,0.254240,0.213864,0.422133
FM (factorization machine),0.292217,0.324054,0.257375,0.482514


## Figure for the report

Grouped bars for all five metrics on one axis -- Recall@3, NDCG@3, Precision@3, MRR (per-decider
ranking quality) and Coverage@3 (a population-level quantity, not a per-decider one, but on the
same 0-1 scale) -- so all five sit side by side for direct comparison instead of Coverage@3 being
split onto its own panel. Error bars are the across-fold standard deviation.


In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
BASELINE_COLOR = "#c3c2b7"
SURFACE = "#fcfcfb"
SERIES = {
    "Z (popularity baseline)": "#2a78d6",
    "X1 (clustering)": "#eb6834",
    "FM (factorization machine)": "#1baf7a",
}
METHOD_ORDER = list(SERIES.keys())
SHORT_NAME = {
    "Z (popularity baseline)": "Z (baseline)",
    "X1 (clustering)": "X1 (clustering)",
    "FM (factorization machine)": "FM",
}
METRICS = [f"Recall@{K}", f"NDCG@{K}", f"Precision@{K}", "MRR", f"Coverage@{K}"]

plt.rcParams["font.family"] = "DejaVu Sans"
fig, ax = plt.subplots(1, 1, figsize=(9.5, 4.8))
fig.patch.set_facecolor(SURFACE)

ax.set_facecolor(SURFACE)
n_methods = len(METHOD_ORDER)
group_width = 0.72
bar_width = group_width / n_methods
x = np.arange(len(METRICS))
for i, method in enumerate(METHOD_ORDER):
    means = table.loc[method, [f"{m}_mean" for m in METRICS]].to_numpy(dtype=float)
    stds = table.loc[method, [f"{m}_std" for m in METRICS]].to_numpy(dtype=float)
    offset = (i - (n_methods - 1) / 2) * bar_width
    ax.bar(x + offset, means, width=bar_width * 0.88, color=SERIES[method],
           label=SHORT_NAME[method], edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.errorbar(x + offset, means, yerr=stds, fmt="none", ecolor=INK_MUTED,
                elinewidth=1, capsize=2, zorder=4)
    for xi, m in zip(x + offset, means):
        ax.text(xi, m + 0.012, f"{m:.2f}", ha="center", va="bottom", fontsize=8, color=INK_SECONDARY)
ax.set_xticks(x)
ax.set_xticklabels(METRICS, color=INK_PRIMARY, fontsize=10)
ax.set_ylabel("Score (mean over 21 folds)", color=INK_SECONDARY, fontsize=9)
ymax = max(0.62, table[[f"{m}_mean" for m in METRICS]].to_numpy(dtype=float).max()
           + table[[f"{m}_std" for m in METRICS]].to_numpy(dtype=float).max() + 0.08)
ax.set_ylim(0, ymax)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
ax.grid(axis="y", color=GRIDLINE, linewidth=1, zorder=0)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(BASELINE_COLOR)
ax.tick_params(axis="y", colors=INK_MUTED, labelsize=8, length=0)
ax.tick_params(axis="x", colors=INK_PRIMARY, length=0)
ax.set_title("Evaluation metrics @3, leave-one-wave-out CV (21 folds)",
              color=INK_PRIMARY, fontsize=10.5, loc="left", pad=12)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.18), ncol=3, frameon=False,
          fontsize=9, labelcolor=INK_SECONDARY, handlelength=1.1, handleheight=1.1,
          columnspacing=1.3)

fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig("figures/comparison_results.png", dpi=200, facecolor=SURFACE)
plt.show()
print("Saved figures/comparison_results.png")


Saved figures/comparison_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_17776\969945038.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Takeaways for the Results section

- **Z (popularity baseline)** and **FM** both beat a random ranking reliably
  (p < 0.05, both tests); **X1 (clustering)** does not.
- **FM vs. Z**: FM is numerically ahead on every metric, but the difference is not
  statistically significant across the 21 folds (paired t-test p = 0.44,
  Wilcoxon p = 0.23) -- report this as "comparable to, with a small edge over,"
  not as a clear win.
- **Coverage@3** tells a different story: Z only ever recommends from a narrow
  slice of the catalogue (0.28), while X1 and FM both reach roughly two-thirds of
  candidates (0.65) -- even though X1's *ranking quality* is the weakest of the
  three, it is not simply recommending the same few popular people over and over.
- All numbers, the saved table (`comparison_results.csv`) and the figure
  (`figures/comparison_results.png`) are reproducible from this notebook alone,
  using only each method's own module (`baseline_desirability.py`,
  `x1_cluster.py`, `sd_fm.py`) and the shared `evaluation.py`.
